In [1]:
# 3.1 Import Library
import pandas as pd
import numpy as np
import re
import os
import ast
import json

In [2]:
# 3.2 Konfigurasi Path Input/Output
INPUT_FILE = '../../../no_stemming/outputs/SLA/lexicon_matching_with_ignore.csv'
OUTPUT_FILE = '../../../no_stemming/outputs/SLA/vader_heuristics_with_ignore.csv'
STATS_FILE = '../../../no_stemming/outputs/SLA/heuristics_stats_with_ignore.json'
DETAIL_FILE = '../../../no_stemming/outputs/SLA/heuristics_detail_with_ignore.csv'
SLA_LEXICON_PATH = '../../../no_stemming/outputs/SLA/sla_lexicon_adapted.csv'
POLITIK_PATH = '../../../kamus/inset_vader_political_modified.csv'

print(f"Input          : {INPUT_FILE}")
print(f"Output         : {OUTPUT_FILE}")
print(f"Stats          : {STATS_FILE}")
print(f"SLA Lexicon    : {SLA_LEXICON_PATH}")

Input          : ../../../no_stemming/outputs/SLA/lexicon_matching_with_ignore.csv
Output         : ../../../no_stemming/outputs/SLA/vader_heuristics_with_ignore.csv
Stats          : ../../../no_stemming/outputs/SLA/heuristics_stats_with_ignore.json
SLA Lexicon    : ../../../no_stemming/outputs/SLA/sla_lexicon_adapted.csv


In [3]:
# 3.3 Load Data dan Rekonstruksi Word Scores
df = pd.read_csv(INPUT_FILE)

# 1. Load kedua kamus
df_inset_sla = pd.read_csv(SLA_LEXICON_PATH)
df_politik_sla = pd.read_csv(POLITIK_PATH)

# 2. Gabungkan
df_combined_lex = pd.concat([df_inset_sla, df_politik_sla]).drop_duplicates(subset='kata', keep='last')
lexicon_dict = dict(zip(df_combined_lex['kata'].astype(str).str.lower(), df_combined_lex['mean']))

# 3. Konversi kolom list
list_cols = ['tokens', 'matched_words', 'unmatched_words', 'filtered_tokens', 'filtered_word_scores']
for col in list_cols:
    if col in df.columns:
        df[col] = df[col].apply(ast.literal_eval)

# 4. Fungsi Rekonstruksi 
def reconstruct_word_scores(tokens, lexicon):
    details = []
    for tok in tokens:
        t_low = tok.lower()
        score = lexicon.get(t_low, 0)
        details.append({'word': tok, 'score': score, 'matched': t_low in lexicon})
    return details

# Pilih kolom token sesuai skenario 
token_col = 'filtered_tokens' if 'filtered_tokens' in df.columns else 'tokens'

df['word_scores_detail'] = df.apply(
    lambda row: reconstruct_word_scores(row[token_col], lexicon_dict), axis=1
)

In [4]:
# 3.4 Parameter Heuristik VADER (SINKRON DENGAN PAPER & RSN)
PUNCT_BOOST = 0.292  # [cite: 237]
MAX_PUNCT_BOOST = 1.168  # VADER membatasi hingga 4 tanda seru [cite: 237]

CAP_BOOST = 0.733 # [cite: 233]
CAP_THRESHOLD = 0.5

# Nilai standar VADER asli (B_INCR = 0.293) [cite: 233]
B_INCR = 0.293
B_DECR = -0.293

BOOSTERS = {
    # Penguat (Boosters) - Menambah intensitas sebesar 0.293
    'sangat': B_INCR, 'sekali': B_INCR, 'banget': B_INCR, 'amat': B_INCR, 'teramat': B_INCR,
    'terlalu': B_INCR, 'benar': B_INCR, 'betul': B_INCR, 'pasti': B_INCR, 'jelas': B_INCR,
    'nyata': B_INCR, 'parah': B_INCR, 'gila': B_INCR, 'luar biasa': B_INCR,

    # Pelemah (Dampeners) - Mengurangi intensitas sebesar 0.293
    'agak': B_DECR, 'sedikit': B_DECR, 'kurang': B_DECR, 'hampir': B_DECR, 'nyaris': B_DECR,
    'hanya': B_DECR, 'cuma': B_DECR, 'sekadar': B_DECR, 'mungkin': B_DECR, 'sepertinya': B_DECR,
    'lumayan': B_DECR, 'cukup': B_DECR
}

# Aturan Kata Hubung Kontras (But Check) [cite: 238, 239]
CONTRAST_CONJUNCTIONS = ['tapi', 'tetapi', 'namun', 'walaupun', 'meskipun', 'kendati',
                         'sekalipun', 'sedangkan', 'padahal', 'sementara', 'biarpun', 'walau', 'meski']
PRE_BUT_SCALAR = 0.5  # [cite: 239]
POST_BUT_SCALAR = 1.5  # [cite: 239]

# Aturan Negasi [cite: 209, 210]
NEGATIONS = ['tidak', 'tak', 'nggak', 'gak', 'bukan', 'jangan', 'tanpa', 'belum', 'no', 'never']
NEGATION_WINDOW = 3
NEGATION_SCALAR = -0.74 # Standar VADER [cite: 209]

# Konstanta normalisasi VADER asli
ALPHA = 15.0

# Cetak ringkasan parameter lengkap
print(f"\n[SINKRONISASI] Parameter heuristik VADER berhasil disesuaikan dengan Kode Asli:")
print(f"  - Punctuation boost: {PUNCT_BOOST} (max cap: {MAX_PUNCT_BOOST})")
print(f"  - Capitalization boost factor: {CAP_BOOST} (threshold: {CAP_THRESHOLD})")
print(f"  - Degree modifiers: {len(BOOSTERS)} kata (Incr/Decr: {B_INCR})")
print(f"  - Contrast conjunctions: {len(CONTRAST_CONJUNCTIONS)} kata (Pre: {PRE_BUT_SCALAR}, Post: {POST_BUT_SCALAR})")
print(f"  - Negations: {len(NEGATIONS)} kata (Scalar: {NEGATION_SCALAR})")



[SINKRONISASI] Parameter heuristik VADER berhasil disesuaikan dengan Kode Asli:
  - Punctuation boost: 0.292 (max cap: 1.168)
  - Capitalization boost factor: 0.733 (threshold: 0.5)
  - Degree modifiers: 26 kata (Incr/Decr: 0.293)
  - Contrast conjunctions: 13 kata (Pre: 0.5, Post: 1.5)
  - Negations: 10 kata (Scalar: -0.74)


In [5]:
# 3.5 Fungsi Heuristic 1: Punctuation
def apply_punctuation_heuristic(text, base_score):
    if not isinstance(text, str):
        return base_score, {'exclamation_count': 0, 'question_count': 0, 'punct_boost': 0}
    
    exclamation_count = text.count('!')
    question_count = text.count('?')
    
    punct_boost = min(exclamation_count * PUNCT_BOOST, MAX_PUNCT_BOOST)
    question_effect = question_count * -0.1
    total_boost = punct_boost + question_effect
    
    return base_score + total_boost, {
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'punct_boost': total_boost
    }

print("Fungsi Heuristic 1 (Punctuation) siap")

Fungsi Heuristic 1 (Punctuation) siap


In [6]:
# 3.6 Fungsi Heuristic 2: Capitalization
def apply_capitalization_heuristic(tokens, base_score):
    if not tokens:
        return base_score, {'caps_ratio': 0, 'cap_boost': 0}
    
    # Identifikasi kata kapital (panjang minimal 2 huruf untuk menghindari inisial/kata ganti)
    caps_words = [t for t in tokens if t.isupper() and len(t) >= 2]
    alpha_words = [t for t in tokens if t.isalpha()]
    
    total_alpha = len(alpha_words)
    if total_alpha == 0:
        return base_score, {'caps_ratio': 0, 'cap_boost': 0}
    
    cap_ratio = len(caps_words) / total_alpha
    
    # Jika rasio kata kapital melewati ambang batas (threshold)
    if cap_ratio >= CAP_THRESHOLD and base_score != 0:
        # Tentukan arah boost: Jika skor dasar negatif, boost harus negatif. 
        # Jika positif, boost harus positif.
        direction = 1 if base_score > 0 else -1
        cap_boost = CAP_BOOST * direction
        
        return base_score + cap_boost, {'caps_ratio': cap_ratio, 'cap_boost': cap_boost}
    
    return base_score, {'caps_ratio': cap_ratio, 'cap_boost': 0}

print("Fungsi Heuristic 2 (Capitalization) siap")

Fungsi Heuristic 2 (Capitalization) siap


In [7]:
# 3.7 Fungsi Heuristic 3: Degree Modifiers
def apply_degree_modifier_heuristic(tokens, word_scores, base_score):
    adjustments = []
    modified_score = base_score
    
    # Peta indeks kata yang memiliki skor dalam leksikon
    score_map = {i: w['score'] for i, w in enumerate(word_scores) if w['matched']}
    
    for i, token in enumerate(tokens):
        token_lower = token.lower()
        
        if token_lower in BOOSTERS:
            # VADER mencari kata yang dimodifikasi dalam 3 kata ke depan (window of 3)
            for j in range(i + 1, min(i + 4, len(tokens))):
                if j in score_map:
                    # Ambil nilai penambah/pengurang (B_INCR = 0.293 atau B_DECR = -0.293)
                    scalar = BOOSTERS[token_lower]
                    
                    # Jika kata yang dimodifikasi bermakna negatif, arah scalar dibalik
                    # Contoh: "sangat buruk" -> -3 + (-0.293) = -3.293
                    if score_map[j] < 0:
                        scalar *= -1
                        
                    modified_score += scalar
                    
                    adjustments.append({
                        'modifier': token,
                        'target_word': tokens[j],
                        'scalar_applied': scalar,
                        'positions': (i, j)
                    })
                    break # Berhenti setelah menemukan kata pertama yang dimodifikasi
                    
    return modified_score, adjustments

print("Fungsi Heuristic 3 (Degree Modifiers) siap")

Fungsi Heuristic 3 (Degree Modifiers) siap


In [8]:
# 3.8 Fungsi Heuristic 4: Polarity Shift (But Check)
def apply_polarity_shift_heuristic(tokens):
    """
    Mendeteksi keberadaan kata hubung kontras dan menentukan posisinya.
    Logika: Sentimen sebelum 'tapi' dikali 0.5, sesudah 'tapi' dikali 1.5.
    """
    found_conj = None
    conj_index = -1
    
    for i, token in enumerate(tokens):
        if token.lower() in CONTRAST_CONJUNCTIONS:
            found_conj = token.lower()
            conj_index = i
            break # Ambil kata hubung pertama yang ditemukan
            
    if not found_conj:
        return None, None
    
    return conj_index, {'conjunction': found_conj, 'pre_scalar': 0.5, 'post_scalar': 1.5}

print("Fungsi Heuristic 4 (Polarity Shift) siap")

Fungsi Heuristic 4 (Polarity Shift) siap


In [9]:
# 3.9 Fungsi Heuristic 5: Negation
def apply_negation_heuristic(tokens, word_scores):
    total_effect = 0
    adjustments = []
    negated_indices = set()
    
    for i, token in enumerate(tokens):
        if token.lower() in NEGATIONS:
            for j in range(i + 1, min(i + 1 + NEGATION_WINDOW, len(tokens))):
                if j in negated_indices:
                    continue
                
                if word_scores[j]['matched']:
                    original_score = word_scores[j]['score']
                    negated_score = original_score * NEGATION_SCALAR
                    effect = negated_score - original_score
                    
                    total_effect += effect
                    
                    adjustments.append({
                        'negation': token,
                        'negated_word': tokens[j],
                        'original_score': original_score,
                        'negated_score': negated_score,
                        'effect': effect,
                        'positions': (i, j)
                    })
                    
                    negated_indices.add(j)
                    break
                    
    return total_effect, adjustments

print("Fungsi Heuristic 5 (Negation) siap")

Fungsi Heuristic 5 (Negation) siap


In [10]:
# 3.10 Implementasi SLA + 5 Heuristik VADER
print("\nMENERAPKAN STRULTURAL LEXICON ADAPTATION (SLA) + 5 HEURISTIK VADER")

results = []

for idx, row in df.iterrows():
    if (idx + 1) % 2000 == 0:
        print(f"Processing tweet {idx + 1}/{len(df)}")
    
    text = row['teks']
    tokens = row['tokens']
    word_scores = row['word_scores_detail']
    
    # 1. Agregasi skor dari leksikon SLA
    base_score_sla = sum(w['score'] for w in word_scores if w['matched'])

    # 2. Skor awal sebelum heuristik
    current_score = base_score_sla

    # 3. Penerapan 5 Heuristik VADER secara berurutan
    
    # H1: Punctuation
    score_after_punct, punct_info = apply_punctuation_heuristic(text, current_score)
    
    # H2: Capitalization
    score_after_caps, caps_info = apply_capitalization_heuristic(tokens, score_after_punct)
    
    # H3: Degree Modifiers
    score_after_modifiers, mod_adj = apply_degree_modifier_heuristic(tokens, word_scores, score_after_caps)

    # H4: Polarity Shift (But Check)
    conj_index, shift_info = apply_polarity_shift_heuristic(tokens)
    if conj_index is not None:
        score_after_shift = score_after_modifiers * 1.5
    else:
        score_after_shift = score_after_modifiers

    # H5: Negation
    neg_effect, neg_adj = apply_negation_heuristic(tokens, word_scores)
    final_raw_score = score_after_shift + neg_effect

    # 4. Transformasi ke Compound Score [-1, 1]
    compound_score = final_raw_score / np.sqrt(final_raw_score**2 + ALPHA)
    compound_score = np.clip(compound_score, -1.0, 1.0)
    
    # 5. Klasifikasi Sentimen (Threshold ±0.05)
    if compound_score > 0.05:
        sentiment = 'positive'
    elif compound_score < -0.05:
        sentiment = 'negative'
    else:
        sentiment = 'neutral'
    
    results.append({
        'no': row['no'],
        'timestamp': row['timestamp'],
        'teks': text,
        'teks_processed': row['teks_processed'],
        'base_score_sla': base_score_sla,
        'score_after_punctuation': score_after_punct,
        'score_after_capitalization': score_after_caps,
        'score_after_modifiers': score_after_modifiers,
        'score_after_polarity_shift': score_after_shift,
        'final_raw_score': final_raw_score,
        'compound_score': compound_score,
        'sentiment': sentiment,
        'heuristics_detail': {
            'punctuation': punct_info,
            'capitalization': caps_info,
            'degree_modifiers': mod_adj,
            'polarity_shift': shift_info,
            'negation': neg_adj
        }
    })

print(f"\nSelesai memproses {len(results)} tweet.")


MENERAPKAN STRULTURAL LEXICON ADAPTATION (SLA) + 5 HEURISTIK VADER
Processing tweet 2000/13192
Processing tweet 4000/13192
Processing tweet 6000/13192
Processing tweet 8000/13192
Processing tweet 10000/13192
Processing tweet 12000/13192

Selesai memproses 13192 tweet.


In [11]:
# 3.11 Buat DataFrame Hasil
df_results = pd.DataFrame(results)
df_to_save = df_results.drop(columns=['heuristics_detail'])
print(f"DataFrame hasil dibuat: {len(df_results)} rows")

DataFrame hasil dibuat: 13192 rows


In [12]:
# 3.12 Hitung Statistik Heuristik
stats = {
    'total_tweets': len(df_results),
    'sentiment_distribution': df_results['sentiment'].value_counts().to_dict(),
    'compound_score_stats': {
        'mean': float(df_results['compound_score'].mean()),
        'std': float(df_results['compound_score'].std()),
        'min': float(df_results['compound_score'].min()),
        'max': float(df_results['compound_score'].max())
    },
    'heuristic_usage': {
        'punctuation_applied': sum(1 for r in results if r['heuristics_detail']['punctuation']['exclamation_count'] > 0),
        'capitalization_applied': sum(1 for r in results if r['heuristics_detail']['capitalization']['cap_boost'] > 0),
        'degree_modifiers_applied': sum(1 for r in results if len(r['heuristics_detail']['degree_modifiers']) > 0),
        'polarity_shift_applied': sum(1 for r in results if r['heuristics_detail']['polarity_shift'] is not None),
        'negation_applied': sum(1 for r in results if len(r['heuristics_detail']['negation']) > 0)
    }
}

print("\nSTATISTIK HEURISTIK VADER")
print("\nDistribusi Sentimen:")
for sent, count in stats['sentiment_distribution'].items():
    pct = count / len(df_results) * 100
    print(f"  {sent}: {count} ({pct:.2f}%)")

print(f"\nCompound Score:")
print(f"  Mean: {stats['compound_score_stats']['mean']:.4f}")
print(f"  Std:  {stats['compound_score_stats']['std']:.4f}")
print(f"  Min:  {stats['compound_score_stats']['min']:.4f}")
print(f"  Max:  {stats['compound_score_stats']['max']:.4f}")

print(f"\nHeuristik yang Diterapkan:")
print(f"  Punctuation:      {stats['heuristic_usage']['punctuation_applied']} tweet")
print(f"  Capitalization:   {stats['heuristic_usage']['capitalization_applied']} tweet")
print(f"  Degree Modifiers: {stats['heuristic_usage']['degree_modifiers_applied']} tweet")
print(f"  Polarity Shift:   {stats['heuristic_usage']['polarity_shift_applied']} tweet")
print(f"  Negation:         {stats['heuristic_usage']['negation_applied']} tweet")


STATISTIK HEURISTIK VADER

Distribusi Sentimen:
  negative: 6479 (49.11%)
  positive: 6318 (47.89%)
  neutral: 395 (2.99%)

Compound Score:
  Mean: -0.0365
  Std:  0.6796
  Min:  -0.9972
  Max:  0.9902

Heuristik yang Diterapkan:
  Punctuation:      606 tweet
  Capitalization:   191 tweet
  Degree Modifiers: 941 tweet
  Polarity Shift:   622 tweet
  Negation:         2220 tweet


In [13]:
# 3.13 Simpan Hasil
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df_to_save.to_csv(OUTPUT_FILE, index=False)
print(f"\nHasil utama disimpan: {OUTPUT_FILE}")

with open(STATS_FILE, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"Statistik disimpan: {STATS_FILE}")

heuristics_df = pd.DataFrame([r['heuristics_detail'] for r in results])
heuristics_df['no'] = [r['no'] for r in results]
heuristics_df.to_csv(DETAIL_FILE, index=False)
print(f"Detail heuristik disimpan: {DETAIL_FILE}")


Hasil utama disimpan: ../../../no_stemming/outputs/SLA/vader_heuristics_with_ignore.csv
Statistik disimpan: ../../../no_stemming/outputs/SLA/heuristics_stats_with_ignore.json
Detail heuristik disimpan: ../../../no_stemming/outputs/SLA/heuristics_detail_with_ignore.csv


In [14]:
# 3.14 Preview Hasil
print("\nPREVIEW HASIL ANALISIS SENTIMEN")
for i in [0, 1]:
    print(f"\nTweet {df_results.iloc[i]['no']}:")
    print(f"   Teks: {df_results.iloc[i]['teks_processed'][:80]}...")
    print(f"   Base Score (SLA): {df_results.iloc[i]['base_score_sla']:.4f}")
    print(f"   Compound Score: {df_results.iloc[i]['compound_score']:.4f}")
    print(f"   Sentimen: {df_results.iloc[i]['sentiment'].upper()}")
    
    details = df_results.iloc[i]['heuristics_detail']
    print(f"   Heuristics Applied:")
    if details['punctuation']['exclamation_count'] > 0:
        print(f"     - Punctuation: {details['punctuation']['exclamation_count']} tanda seru")
    if details['capitalization']['cap_boost'] > 0:
        print(f"     - Capitalization: boost {details['capitalization']['cap_boost']:.4f}")
    if details['degree_modifiers']:
        print(f"     - Degree Modifiers: {len(details['degree_modifiers'])} adjustments")
    if details['polarity_shift']:
        print(f"     - Polarity Shift: {details['polarity_shift']['conjunction']}")
    if details['negation']:
        print(f"     - Negation: {len(details['negation'])} words negated")

print(f"\nCek distribusi sentimen:")
print(f"  Positive: {(df_results['sentiment'] == 'positive').sum()}")
print(f"  Neutral:  {(df_results['sentiment'] == 'neutral').sum()}")
print(f"  Negative: {(df_results['sentiment'] == 'negative').sum()}")


PREVIEW HASIL ANALISIS SENTIMEN

Tweet 1:
   Teks: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
   Base Score (SLA): -1.0000
   Compound Score: -0.0320
   Sentimen: NEUTRAL
   Heuristics Applied:
     - Punctuation: 3 tanda seru

Tweet 2:
   Teks: tertibkan media online DPR pemerintah jangan sporadis apalagi selektif hanya kep...
   Base Score (SLA): -12.5000
   Compound Score: -0.9682
   Sentimen: NEGATIVE
   Heuristics Applied:
     - Degree Modifiers: 1 adjustments
     - Negation: 1 words negated

Cek distribusi sentimen:
  Positive: 6318
  Neutral:  395
  Negative: 6479


In [15]:
# 3.15 Simpan Versi Ringkas
CLEAN_OUTPUT_FILE = '../../../no_stemming/outputs/SLA/sentiment_clean_with_ignore.csv'
df_clean = df_results[['no', 'timestamp', 'teks', 'teks_processed', 'sentiment', 'compound_score']]
df_clean.to_csv(CLEAN_OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\nData ringkas berhasil disimpan: {CLEAN_OUTPUT_FILE}")


Data ringkas berhasil disimpan: ../../../no_stemming/outputs/SLA/sentiment_clean_with_ignore.csv
